# All OpenAI Solution

## Installations

In [10]:
!pip install -qq llama-index
!pip install -qq llama-index-llms-azure_openai llama_index.embeddings.jinaai
!pip install -qq llama-index-readers-file
!pip install -qq llama-index-packs-rag-evaluator

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-packs-rag-evaluator 0.4.1 requires llama-index-llms-openai<0.6,>=0.5.0, but you have llama-index-llms-openai 0.6.15 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index 0.14.13 requires llama-index-llms-openai<0.7,>=0.6.0, but you have llama-index-llms-openai 0.5.6 which is incompatible.
llama-index-cli 0.5.3 requires llama-index-llms-openai<0.7,>=0.6.0, but you have llama-index-llms-openai 0.5.6 which is incompatible.
llama-index-llms-azure-openai 0.4.2 requires llama-index-llms-openai<0.7,>=0.6.0, but you have llama-index-llms-openai 0.5.6 which is incompatible.


## Imports

In [11]:
from pathlib import Path
from llama_index.readers.file import PDFReader
import os

from dotenv import load_dotenv, find_dotenv
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.jinaai import JinaEmbedding
from llama_index.core.llms import ChatMessage
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex
from llama_index.core.llms import ChatMessage

from llama_index.core.ingestion import IngestionPipeline

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Response,
)

from llama_index.core.llama_dataset import LabelledRagDataset
from llama_index.packs.rag_evaluator import RagEvaluatorPack
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    CorrectnessEvaluator,
    RetrieverEvaluator,
    generate_question_context_pairs,
    EmbeddingQAFinetuneDataset
)

from llama_index.core.llama_dataset.generator import RagDatasetGenerator

## Environment Variables

In [12]:
# load_dotenv('/home/santhosh/Projects/courses/Pinnacle/.env')

os.environ["AZURE_OPENAI_API_KEY"] = "F8kUozKumg8vOqdM6i3uF3MEnHQyAWnh5si5hgocdPdXbakenhTWJQQJ99BLACfhMk5XJ3w3AAAAACOGSVUE"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://genaifoundry766488650611.openai.azure.com/"
os.environ["OPENAI_API_VERSION"] = "2024-02-01"

## Reading the PDF file

In [13]:
loader = PDFReader()

documents = loader.load_data(file=Path('/content/Final Policy document_LICs New Jeevan Shanti_V05_logo.pdf'))

len(documents)

21

## Creating the Vector Store

In [14]:
embed_model = JinaEmbedding(
    model="jina-embeddings-v3",
    api_key="jina_9802e91e65a04bc19a54a07ef34ed6aas65UwE-1gl4uWCyYQhZys98rZRod",
    task="retrieval.passage",
)

In [15]:
embedding = embed_model.get_text_embedding("The cat sat on the mat")

In [16]:
# https://developers.llamaindex.ai/python/framework/module_guides/indexing/vector_store_index/#using-the-ingestion-pipeline-to-create-nodes

# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=20),
        embed_model,
    ]
)

# run the pipeline
nodes = pipeline.run(documents=documents)

len(nodes)

80

In [17]:
vector_index = VectorStoreIndex(
    nodes,
    show_progress=True,
    embed_model=embed_model,
)

Generating embeddings: 0it [00:00, ?it/s]

In [18]:
vector_retriever = vector_index.as_retriever(similarity_top_k=3)

In [19]:
retrieved_nodes = vector_retriever.retrieve("When should we notify the death certificate?")

In [20]:
retrieved_nodes[1].text

'(7) Upon the receipt of the notice referred to in sub-section (5), the insurer shall record the fact of \nsuch transfer or assignment together with the date thereof and the name of the transferee or the \nassignee and shall, on the request of the person by whom the notice was given, or of  the \ntransferee or assignee, on payment of such fee as may be specified by the regulations, grant a \nwritten acknowledgement of the receipt of such notice; and any such acknowledgement shall be \nconclusive evidence against the insurer that he has duly received the notice to which such \nacknowledgment relates.'

In [21]:
llm = AzureOpenAI(
    engine="gpt-4o-mini",
    model="gpt-4o-mini",
    temperature=0.0,
)

In [22]:
chat_engine = vector_index.as_chat_engine(chat_mode="context", llm=llm)

In [23]:
response = chat_engine.chat("When should we notify the death certificate?")

In [24]:
print(response)

You must notify the death certificate in writing to the office of the Corporation where the policy is serviced within 90 days from the date of death for any claims to be admissible.


# Evaluating the model

In [25]:
llm_judge = AzureOpenAI(
    engine="gpt-4o",
    model="gpt-4o",
    temperature=0.0,
)

In [26]:
data_generator = RagDatasetGenerator.from_documents(
    documents,
    llm=llm_judge,
    num_questions_per_chunk=2
)

In [27]:
eval_dataset = data_generator.generate_dataset_from_nodes()

In [28]:
eval_dataset.examples[0].query

'**Comprehension Question:**'

In [29]:
eval_dataset.examples[0].reference_answer

'Could you please clarify your query or specify the question you would like answered based on the provided context?'

In [30]:
eval_questions = [example.query for example in eval_dataset.examples]
eval_answers = [example.reference_answer for example in eval_dataset.examples]

In [31]:
len(eval_answers)

42

In [32]:
# Query Engine
query_engine = vector_index.as_query_engine(llm=llm)
# Create Evaluators
relevancy_evaluator = RelevancyEvaluator(llm=llm)
faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_judge)
correctness_evaluator = CorrectnessEvaluator(llm=llm_judge)

In [33]:
from llama_index.core.evaluation import BatchEvalRunner

runner = BatchEvalRunner(
    {
     "faithfulness": faithfulness_evaluator,
     "relevancy": relevancy_evaluator,
     "correctness": correctness_evaluator
     },
    workers=8,
)

eval_results = await runner.aevaluate_queries(
    query_engine, queries=eval_questions, reference = eval_answers
)

In [34]:
def get_eval_results(key, eval_results):
    results = eval_results[key]
    correct = 0
    for result in results:
        if result.passing:
            correct += 1
    score = correct / len(results)
    print(f"{key} Score: {score}")
    return score

In [35]:
get_eval_results("faithfulness", eval_results)
get_eval_results("relevancy", eval_results)
_ = get_eval_results("correctness", eval_results)

faithfulness Score: 1.0
relevancy Score: 1.0
correctness Score: 0.7619047619047619
